In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, f1_score, roc_auc_score, confusion_matrix

import torch
from torch import nn

In [23]:
base_path = Path.cwd().parent/"data"/"raw"
file_path = base_path/"LBG Step Up Data Set.xlsx"
df = pd.read_excel(file_path)

In [24]:
loan_status = {'Charged Off': 1, 'Fully Paid': 0}
df['loan_status'] = df['loan_status'].map(loan_status)

X = df.drop('loan_status', axis=1)
y = df['loan_status']

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [25]:
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10
}
columns_to_drop = ['emp_title', 'addr_state', 'id']

def features_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['emp_length'] = df['emp_length'].map(emp_map)
    df = df.drop(columns=columns_to_drop)
    return df

class Winsorize(BaseEstimator, TransformerMixin):
    def __init__(self, lower_quantile=0.01, upper_quantile=0.99):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.lower_bound_ = X.quantile(self.lower_quantile)
        self.upper_bound_ = X.quantile(self.upper_quantile)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X = X.clip(lower=self.lower_bound_, upper=self.upper_bound_, axis=1)
        return X.values

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

X_cleaned = features_clean(X_train)
one_hot_cols = ['home_ownership', 'purpose', 'term']
transform_cols = ['total_bal_ex_mort', 'max_bal_bc', 'avg_cur_bal',
               'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl']
num_cols = X_cleaned.select_dtypes(include=[float, int]).columns.tolist()
non_transformed_cols = [col for col in num_cols if col not in transform_cols]


transform_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('winsorize', Winsorize()),
    ('transform', PowerTransformer())
])

non_transform_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

torch_pipeline = ColumnTransformer(transformers=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), one_hot_cols),
    ('transform_pipe', transform_pipe, transform_cols),
    ('non_transformed_cols', non_transform_pipe, non_transformed_cols)
])

final_pipeline = Pipeline(steps=[
    ('features_clean', FunctionTransformer(features_clean)),
    ('torch_preproceess', torch_pipeline)
])

X_train_torch = final_pipeline.fit_transform(X_train)
X_test_torch = final_pipeline.transform(X_test)


In [26]:
X_train_torch.shape, y_train.shape, X_test_torch.shape, y_test.shape

((14659, 42), (14659,), (3665, 42), (3665,))

In [27]:
X_train_t = torch.from_numpy(X_train_torch).type(torch.float)
X_test_t = torch.from_numpy(X_test_torch).type(torch.float)
y_train_t = torch.from_numpy(y_train.values).type(torch.float)
y_test_t = torch.from_numpy(y_test.values).type(torch.float)

In [28]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train_t, y_train_t, test_size=0.2, stratify=y_train_t, random_state=42)
len(X_tr), len(y_tr), len(X_val), len(y_val)

(11727, 11727, 2932, 2932)

In [29]:
from torch.utils.data import DataLoader, TensorDataset

train_ds = TensorDataset(X_tr, y_tr)
val_ds = TensorDataset(X_val, y_val)

train_dataloader = DataLoader(train_ds,
                              batch_size=16,
                              shuffle=True)
test_dataloader = DataLoader(val_ds, batch_size=16,
                             shuffle=False)

In [30]:
# Let's check out what we've created
print(f'DataLoaders: {train_dataloader, test_dataloader}')
print(f'Length of train dataloader: {len(train_dataloader)} batches of {16}')
print(f'Length of test dataloader : {len(test_dataloader)} batches of {16} ')

DataLoaders: (<torch.utils.data.dataloader.DataLoader object at 0x000001EA51E4FE50>, <torch.utils.data.dataloader.DataLoader object at 0x000001EA52918750>)
Length of train dataloader: 733 batches of 16
Length of test dataloader : 184 batches of 16 


In [31]:
# Check out what's inside the training dataloader
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch.shape, train_labels_batch.shape

(torch.Size([16, 42]), torch.Size([16]))

In [32]:
OUTPUT_SHAPE = 1
INPUT_FEATURES = X_tr.shape[1]
HIDDEN_UNITS = 20
device = "cuda" if torch.cuda.is_available() else "cpu"

In [33]:
torch.manual_seed(42)

class LoanDefaultV0(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int, drop_out=0.2):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Linear(input_shape, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Dropout(drop_out),
            nn.Linear(hidden_units, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Dropout(drop_out),
            nn.Linear(hidden_units, output_shape)
        )
    def forward(self, x):
        return self.layer_stack(x)

model = LoanDefaultV0(input_shape=INPUT_FEATURES,
                      hidden_units=HIDDEN_UNITS,
                      output_shape=OUTPUT_SHAPE).to(device)
model, model.state_dict()

(LoanDefaultV0(
   (layer_stack): Sequential(
     (0): Linear(in_features=42, out_features=20, bias=True)
     (1): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (2): ReLU()
     (3): Dropout(p=0.2, inplace=False)
     (4): Linear(in_features=20, out_features=20, bias=True)
     (5): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (6): ReLU()
     (7): Dropout(p=0.2, inplace=False)
     (8): Linear(in_features=20, out_features=1, bias=True)
   )
 ),
 OrderedDict([('layer_stack.0.weight',
               tensor([[ 0.1180,  0.1281, -0.0361,  0.1417, -0.0338,  0.0311, -0.0751,  0.0906,
                         0.1360, -0.1132,  0.1341,  0.0289,  0.1140,  0.0209,  0.0744, -0.0218,
                         0.1190,  0.0228, -0.0720,  0.0393, -0.0711, -0.0181, -0.0627,  0.1024,
                        -0.1218, -0.0711, -0.0436, -0.0928,  0.0146, -0.1524,  0.1394, -0.1311,
                         0.1191,  0.0257, -0.0

In [34]:
ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
pos_weight = torch.tensor([ratio], dtype=torch.float).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(),
                             lr=1e-3)

In [35]:
from torchmetrics import Accuracy
accuracy_fn = Accuracy(task='binary').to(device)
accuracy_fn

BinaryAccuracy()

In [42]:
# Dummy test
dummy = torch.rand(16, 42, dtype=torch.float).to(device)
dummy_result = torch.round(torch.sigmoid(model(dummy).squeeze()))
dummy_result

tensor([1., 1., 0., 0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1.],
       device='cuda:0', grad_fn=<RoundBackward0>)

In [37]:
y_val

tensor([0., 0., 0.,  ..., 1., 0., 1.])

In [38]:
from tqdm.notebook import tqdm
torch.manual_seed(42)

epochs = 20

for epoch in tqdm(range(epochs)):
    train_loss = 0
    for batch, (X, y) in enumerate(train_dataloader):
        model.train()
        X, y = X.to(device), y.to(device)
        y_logits = model(X).squeeze()
        y_proba = torch.sigmoid(y_logits)
        y_pred = torch.round(y_proba)
        loss = loss_fn(y_logits, y)
        train_loss += loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 400 == 0:
            print(f'Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples')
    train_loss /= len(train_dataloader)

    ### Testing
    # Setup variables for accumulatively adding up loss and accuracy 
    test_loss, test_acc = 0, 0 
    model.eval()
    with torch.inference_mode():
        for X, y in test_dataloader:
            X, y = X.to(device), y.to(device)
            # 1. Forward pass
            test_pred = model(X).squeeze()

            # 2. Calculate loss (accumulatively)
            test_loss += loss_fn(test_pred, y) # accumulatively add up the loss per epoch

            # 3. Calculate accuracy (preds need to be same as y_true)
            test_acc += accuracy_fn(torch.round(torch.sigmoid(test_pred)), y)
        
        # Calculations on test metrics need to happen inside torch.inference_mode()
        # Divide total test loss by length of test dataloader (per batch)
        test_loss /= len(test_dataloader)

        # Divide total accuracy by length of test dataloader (per batch)
        test_acc /= len(test_dataloader)

    ## Print out what's happening
    print(f"\nTrain loss: {train_loss:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%\n")



  0%|          | 0/20 [00:00<?, ?it/s]

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.05090 | Test loss: 1.01360, Test acc: 0.66%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.02808 | Test loss: 1.01319, Test acc: 0.61%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.01779 | Test loss: 1.01110, Test acc: 0.61%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.01564 | Test loss: 1.01433, Test acc: 0.61%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.00834 | Test loss: 1.01814, Test acc: 0.59%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.00540 | Test loss: 1.01851, Test acc: 0.63%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.00817 | Test loss: 1.01719, Test acc: 0.64%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.00496 | Test loss: 1.01349, Test acc: 0.64%

Looked at 0/11727 samples
Looked at 6400/11727 samples

Train loss: 1.00190 | Te

In [40]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

model.eval()
all_preds, all_probs, all_targets = [], [], []
with torch.inference_mode():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        logits = model(X).squeeze(dim=1)
        probs = torch.sigmoid(logits)
        preds = torch.round(probs)
        all_preds.append(preds)
        all_probs.append(probs)
        all_targets.append(y)

all_preds = torch.cat(all_preds).cpu()
all_probs = torch.cat(all_probs).cpu()
all_targets = torch.cat(all_targets).cpu()

print(confusion_matrix(all_targets, all_preds))
print(classification_report(all_targets, all_preds))
print("AUC:", roc_auc_score(all_targets, all_probs))

[[1568  739]
 [ 265  360]]
              precision    recall  f1-score   support

         0.0       0.86      0.68      0.76      2307
         1.0       0.33      0.58      0.42       625

    accuracy                           0.66      2932
   macro avg       0.59      0.63      0.59      2932
weighted avg       0.74      0.66      0.69      2932

AUC: 0.6821895101863893
